In [ ]:
!pip install -q -U transformers peft trl datasets accelerate "torchao>=0.16.0" bitsandbytes liger-kernel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 141.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 113.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 43.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import torch
import warnings
import ast
from google.colab import drive
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings("ignore")

# 1. Kết nối Google Drive
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/Chef LLM/Weight'
os.makedirs(SAVE_DIR, exist_ok=True)
CSV_DATA_PATH = '/content/drive/MyDrive/Chef LLM/recipes.csv'

# 2. Cấu hình Model & Tối ưu hóa phần cứng
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading model in 4-bit (QLoRA) with SDPA...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
    attn_implementation="sdpa",
    trust_remote_code=True
)
model.config.use_cache = False

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# 3. Cấu hình LoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Mounted at /content/drive


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [ ]:
# 4. Tải và tiền xử lý Dataset
print(f"Đang tải dataset từ {CSV_DATA_PATH}...")
if not os.path.exists(CSV_DATA_PATH):
    raise FileNotFoundError(f"Không tìm thấy file tại {CSV_DATA_PATH}. Kiểm tra lại đường dẫn!")

full_dataset = load_dataset('csv', data_files=CSV_DATA_PATH, split="train")
dataset = full_dataset.shuffle(seed=42).select(range(len(full_dataset)))

def parse_r_vector(data):
    if not isinstance(data, str): return []
    if data.startswith('c(') and data.endswith(')'):
        items = re.findall(r'"(.*?)"', data[2:-1])
        return [item.replace('""', '').strip() for item in items if item.strip()]
    try:
        parsed = ast.literal_eval(data)
        if isinstance(parsed, list): return parsed
    except: pass
    return [data]

def format_recipe_batch(batch):
    texts = []
    for name, qtys, parts, dirs in zip(batch['Name'], batch['RecipeIngredientQuantities'], batch['RecipeIngredientParts'], batch['RecipeInstructions']):
        if not name or not isinstance(name, str):
            texts.append("")
            continue

        qty_list = parse_r_vector(qtys)
        part_list = parse_r_vector(parts)
        dirs_list = parse_r_vector(dirs)

        combined_ings = []
        max_len = max(len(qty_list), len(part_list))
        for i in range(max_len):
            q = qty_list[i] if i < len(qty_list) else ""
            p = part_list[i] if i < len(part_list) else ""
            combined_ings.append(f"{q} {p}".strip())

        ing_str = "\n- " + "\n- ".join(combined_ings) if combined_ings else "\n- (Không có dữ liệu nguyên liệu)"
        dir_str = "\n".join([f"{i+1}. {step}" for i, step in enumerate(dirs_list)]) if dirs_list else "(Không có dữ liệu hướng dẫn)"

        messages = [
            {"role": "system", "content": "Bạn là một siêu đầu bếp. Người dùng sẽ cung cấp nguyên liệu hoặc một món ăn, nhiệm vụ của bạn là hướng dẫn họ cách nấu chi tiết và ngon nhất."},
            {"role": "user", "content": f"Hãy hướng dẫn tôi nấu món: {name}"},
            {"role": "assistant", "content": f"Chắc chắn rồi, dưới đây là công thức chi tiết cho món {name}:\n\n**Nguyên liệu:**{ing_str}\n\n**Các bước thực hiện:**\n{dir_str}"}
        ]
        texts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False))
    return {"text": texts}

print("Đang format dữ liệu sang chuẩn ChatML...")
dataset = dataset.map(
    format_recipe_batch,
    batched=True,
    batch_size=1000,
    num_proc=os.cpu_count(),
    remove_columns=dataset.column_names
)
dataset = dataset.filter(lambda x: len(x['text']) > 0)
print(f"Tổng số lượng dữ liệu đưa vào huấn luyện: {len(dataset)} mẫu")

Đang tải dataset từ /content/drive/MyDrive/Chef LLM/recipes.csv...


Generating train split: 0 examples [00:00, ? examples/s]

Đang format dữ liệu sang chuẩn ChatML...


Map (num_proc=12):   0%|          | 0/1048543 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1048543 [00:00<?, ? examples/s]

Tổng số lượng dữ liệu đưa vào huấn luyện: 1228 mẫu


In [ ]:
model.train() # Chuyển lại chế độ train

training_args = SFTConfig(
    output_dir=SAVE_DIR,
    dataset_text_field="text",
    max_length=1024,
    packing=False,
    num_train_epochs=6,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    optim="paged_adamw_8bit",
    logging_steps=1,
    save_strategy="epoch",
    learning_rate=5e-5,                  # LR tối đa (lr_max) sau khi warmup xong

    # --- BẮT ĐẦU CẤU HÌNH SCHEDULER ---

    # Tùy chọn 1: "cosine" (Khuyên dùng nhất cho LLM - Mượt mà, hội tụ sâu)
    # Tùy chọn 2: "linear" (LR giảm dần thành một đường thẳng, ổn định)
    # Tùy chọn 3: "cosine_with_restarts" (Dùng khi model hay bị kẹt ở Local Minima)
    lr_scheduler_type="cosine",

    # Bạn có 2 cách để cấu hình giai đoạn Warmup:
    # Cách A (Dùng tỷ lệ): warmup_ratio=0.1 (Nghĩa là 10% số step đầu tiên dùng để warmup)
    # Cách B (Dùng step cụ thể): warmup_steps=20 (Khuyên dùng nếu dataset của bạn nhỏ và bạn biết chính xác số step)
    warmup_ratio=0.1,

    # --- KẾT THÚC CẤU HÌNH SCHEDULER ---

    bf16=True,
    tf32=True,
    torch_compile=False,
    max_grad_norm=1.0,                   # Gradient Clipping: Cắt gọt gradient nếu nó lớn hơn 1.0 (chống nổ gradient)
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Bắt đầu huấn luyện...")
trainer.train()

print(f"Đang lưu adapter vào {SAVE_DIR}...")
trainer.model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Hoàn tất! Bạn có thể kiểm tra Google Drive.")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Bắt đầu huấn luyện...


Step,Training Loss
1,1.008257
2,0.809765
3,1.164070
4,0.869828
5,0.952481
6,0.888265
7,0.953689
8,0.950996
9,0.820651
10,0.715493


In [ ]:
# --- 5. TEST MODEL TRƯỚC KHI TRAIN (BASELINE) ---
print("\n" + "="*50)
print("TEST MODEL TRƯỚC KHI TRAIN (BASELINE):")
test_messages = [
    {"role": "system", "content": "Bạn là một siêu đầu bếp. Người dùng sẽ cung cấp nguyên liệu hoặc một món ăn, nhiệm vụ của bạn là hướng dẫn họ cách nấu chi tiết và ngon nhất."},
    {"role": "user", "content": f"Hãy hướng dẫn tôi nấu ăn món khoai tây nghiền."}
]

test_prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        use_cache=True,
        repetition_penalty=1.15,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(f"\nĐầu bếp AI phản hồi:\n{response}")
print("="*50 + "\n")